In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/mkashifn/nbaiot-dataset/7.gafgyt.combo.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/9.gafgyt.combo.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/5.gafgyt.combo.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/1.mirai.udp.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/4.gafgyt.udp.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/6.gafgyt.udp.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/6.gafgyt.junk.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/data_summary.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/5.gafgyt.udp.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/9.gafgyt.junk.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/9.mirai.scan.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/1.benign.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/2.mirai.udpplain.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/3.gafgyt.combo.csv
/kaggle/input/datasets/mkashifn/nbaiot-dataset/4.gafgyt.combo.csv
/kaggle/input/datasets/mkashi

In [3]:
# ===== CELL 1: Imports, Configuration, and Prior Results =====
 
import os, gc, time, random, warnings, json, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from pathlib import Path
 
warnings.filterwarnings("ignore")
 
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAVE_PLOT = True
except ImportError:
    HAVE_PLOT = False
 
try:
    from scipy import stats as sp_stats
    HAVE_SCIPY = True
except ImportError:
    HAVE_SCIPY = False
 
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    if torch.cuda.is_available():
        torch.cuda.synchronize()
 
# ── Configuration (same as Session 1) ────────────────────────────────────────
CFG = {
    "DATA_PATH":         "/kaggle/input/nbaiot-dataset",
    "SAMPLE_PER_DEVICE": 50_000,
    "FL_ROUNDS":         30,
    "LOCAL_EPOCHS":      2,
    "BATCH_SIZE":        512,
    "LR":                0.001,
    "GM_MAX_ITERS":      100,
    "GM_TOL":            1e-5,
    "GM_NU":             1e-6,
    "TRAIN_RATIO":       0.70,
    "VAL_RATIO":         0.15,
    "TEST_RATIO":        0.15,
    "SEEDS":             [42, 123, 456, 789, 1011, 1415, 1617, 1819, 2021],
    "BYZ_RATIO":         0.40,
    "OUT_DIR":           "/kaggle/working",
}
 
# [FIX-1] ACCCS warmup=3 — CRITICAL CONSISTENCY FIX
# The main ACCCS-GM experiment (acccs_gm_FIXED.py) used warmup_rounds=3.
# Original sensitivity notebook used warmup=1 silently.
# All sensitivity runs must use warmup=3 to be comparable to the main result.
ACCCS_WARMUP = 3
 
# Experiment grids (Session 2 only)
EXP5_GAMMA_VALS = [1.0, 2.0, 5.0, 10.0, 20.0]   # 45 runs, ~2.44h
EXP6_DECAY_VALS = [0.90, 0.95, 0.99, 1.0]         # 36 runs, ~1.95h
# Total: 81 runs × ~195s = ~4.39h
 
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FEAT_COLS = None
INPUT_DIM = None
 
# ── Per-seed reference values (from prior CSVs) ───────────────────────────────
TRUST_GM_40PCT_PER_SEED = {
    42: 0.256737, 123: 0.019376, 456: 0.087194, 789: 0.080735,
    1011: 0.050056, 1415: 0.067817, 1617: 0.048441, 1819: 0.016147, 2021: 0.029065,
}
GM_40PCT_PER_SEED = {
    42: 7.844214, 123: 0.134020, 456: 0.322940, 789: 2.877396,
    1011: 0.406904, 1415: 4.881239, 1617: 0.581292, 1819: 0.295490, 2021: 0.348775,
}
ACCCS_GM_40PCT_PER_SEED = {   # ACCCS-GM (gamma=5, decay=0.95, warmup=3) from main exp
    42: 0.025835, 123: 0.019376, 456: 0.019376, 789: 0.040368,
    1011: 0.030679, 1415: 0.022606, 1617: 0.022606, 1819: 0.020991, 2021: 0.022606,
}
 
# ── Load Session 1 results ────────────────────────────────────────────────────
# Session 1 CSV is required before running this notebook.
_s1_candidates = [
    "/kaggle/input/sensitivity-session1/sensitivity_session1.csv",
    "/kaggle/input/sensitivity_session1/sensitivity_session1.csv",
    "/kaggle/input/sensitivity-session-1/sensitivity_session1.csv",
    "/kaggle/input/datasets/t4xman/sensitivity-session1/sensitivity_session1.csv",
    "/kaggle/working/sensitivity_session1.csv",   # if same Kaggle session
]
SESSION1_DF = None
for _p in _s1_candidates:
    if Path(_p).exists():
        SESSION1_DF = pd.read_csv(_p)
        print(f"✓ Session 1 results loaded: {_p}  ({len(SESSION1_DF)} rows)")
        break
if SESSION1_DF is None:
    print("WARNING: sensitivity_session1.csv NOT FOUND.")
    print("  Add it as a Kaggle dataset from Session 1 output.")
    print("  Combined analysis in Cell 11 will be skipped.")
    SESSION1_DF = pd.DataFrame()
 
n_runs_session2 = (len(EXP5_GAMMA_VALS) + len(EXP6_DECAY_VALS)) * len(CFG["SEEDS"])
print(f"\nDevice     : {DEVICE}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}  : {torch.cuda.get_device_name(i)}")
print(f"Total runs : {n_runs_session2}")
print(f"Est. time  : {n_runs_session2 * 195 / 3600:.2f}h  (195s/run measured)")
print(f"Safe budget: 10.0h  [{'FITS ✓' if n_runs_session2 * 195 / 3600 <= 10.0 else 'TOO LONG ✗'}]")
print(f"ACCCS warmup: {ACCCS_WARMUP} rounds  [FIX-1: was 1, now matches main experiment]")
print("✓ Cell 1 complete")

✓ Session 1 results loaded: /kaggle/input/datasets/t4xman/sensitivity-session1/sensitivity_session1.csv  (117 rows)

Device     : cuda
  GPU 0  : Tesla T4
  GPU 1  : Tesla T4
Total runs : 81
Est. time  : 4.39h  (195s/run measured)
Safe budget: 10.0h  [FITS ✓]
ACCCS warmup: 3 rounds  [FIX-1: was 1, now matches main experiment]
✓ Cell 1 complete


In [4]:
# ===== CELL 2: Dataset Loading =====
 
_DEVICE_NAMES = {
    "1":"Danmini_Doorbell",     "2":"Ecobee_Thermostat",
    "3":"Ennio_Doorbell",       "4":"Philips_B120N10_Baby_Monitor",
    "5":"Provision_PT_737E_Camera","6":"Provision_PT_838_Camera",
    "7":"Samsung_SNH_1011_N_Webcam","8":"SimpleHome_XCS7_1002_Camera",
    "9":"SimpleHome_XCS7_1003_Camera",
}
_DEVICE_IDS = ["1","2","3","4","5","6","7","8","9"]
_ATTACK_SUFFIXES = [
    "benign","gafgyt.combo","gafgyt.junk","gafgyt.scan","gafgyt.tcp","gafgyt.udp",
    "mirai.ack","mirai.scan","mirai.syn","mirai.udp","mirai.udpplain",
]
_CANDIDATES = [
    "/kaggle/input/nbaiot-dataset","/kaggle/input/mkashifn-nbaiot-dataset",
    "/kaggle/input/N-BaIoT","/kaggle/input/nbaiot","/kaggle/input/n-baiot",
]
 
def _is_nbaiot_root(p):
    p = Path(p)
    return p.is_dir() and any((p/f).exists() for f in ["1.benign.csv","2.benign.csv"])
 
def _resolve_path(cfg_path):
    if _is_nbaiot_root(cfg_path): return cfg_path
    for c in _CANDIDATES:
        if _is_nbaiot_root(c): return c
    ki = Path("/kaggle/input")
    if ki.exists():
        for d in sorted(ki.rglob("*")):
            if d.is_dir() and _is_nbaiot_root(d): return str(d)
    raise RuntimeError("N-BaIoT not found.")
 
def load_nbaiot(base_path, sample_per_device=None, seed=42):
    rng  = np.random.RandomState(seed)
    base = Path(_resolve_path(base_path))
    print(f"Root: {base}")
    all_dfs = []
    for did in _DEVICE_IDS:
        idx = int(did) - 1
        parts = []
        for suf in _ATTACK_SUFFIXES:
            fp = base / f"{did}.{suf}.csv"
            if not fp.exists(): continue
            try: df_p = pd.read_csv(fp, header=0)
            except: continue
            if df_p.shape[1] != 115: continue
            df_p["label"] = 0 if suf=="benign" else 1
            df_p["device_id"] = idx
            parts.append(df_p)
        if not parts: continue
        dev = pd.concat(parts, ignore_index=True); del parts; gc.collect()
        if sample_per_device and len(dev) > sample_per_device:
            ben = dev[dev["label"]==0]; att = dev[dev["label"]==1]
            rb = len(ben)/len(dev)
            nb = max(1,int(sample_per_device*rb)); na = sample_per_device - nb
            sb = ben.sample(n=min(nb,len(ben)), random_state=rng.randint(0,2**31))
            sa = att.sample(n=min(na,len(att)), random_state=rng.randint(0,2**31))
            dev = pd.concat([sb,sa],ignore_index=True)
            del ben,att,sb,sa; gc.collect()
        pct = (dev["label"]==0).sum()/len(dev)*100
        print(f"  Device {did} ({_DEVICE_NAMES[did][:24]:24s}): {len(dev):>7,}  [{pct:.1f}%]")
        all_dfs.append(dev); del dev; gc.collect()
    df = pd.concat(all_dfs,ignore_index=True); del all_dfs; gc.collect()
    fc = [c for c in df.columns if c not in ("label","device_id")]
    df[fc] = df[fc].apply(pd.to_numeric,errors="coerce")
    return df.dropna().reset_index(drop=True)
 
print("Loading N-BaIoT …")
t0 = time.time()
RAW_DF = load_nbaiot(CFG["DATA_PATH"], sample_per_device=CFG["SAMPLE_PER_DEVICE"], seed=42)
FEAT_COLS = [c for c in RAW_DF.columns if c not in ("label","device_id")]
INPUT_DIM = len(FEAT_COLS)
assert INPUT_DIM == 115
print(f"Features: {INPUT_DIM}  |  Load: {(time.time()-t0)/60:.1f} min")
print("✓ Cell 2 complete")

Loading N-BaIoT …
Root: /kaggle/input/datasets/mkashifn/nbaiot-dataset
  Device 1 (Danmini_Doorbell        ):  50,000  [4.9%]
  Device 2 (Ecobee_Thermostat       ):  50,000  [1.6%]
  Device 3 (Ennio_Doorbell          ):  50,000  [11.0%]
  Device 4 (Philips_B120N10_Baby_Mon):  50,000  [16.0%]
  Device 5 (Provision_PT_737E_Camera):  50,000  [7.5%]
  Device 6 (Provision_PT_838_Camera ):  50,000  [11.8%]
  Device 7 (Samsung_SNH_1011_N_Webca):  50,000  [13.9%]
  Device 8 (SimpleHome_XCS7_1002_Cam):  50,000  [5.4%]
  Device 9 (SimpleHome_XCS7_1003_Cam):  50,000  [2.3%]
Features: 115  |  Load: 2.0 min
✓ Cell 2 complete


In [5]:
# ===== CELL 3: Preprocessing (seed=42) =====
 
def split_per_device(df, seed=42):
    tr_l,va_l,te_l = [],[],[]
    for dev in sorted(df["device_id"].unique()):
        sub = df[df["device_id"]==dev]
        try:
            tv,te = train_test_split(sub,test_size=CFG["TEST_RATIO"],random_state=seed,stratify=sub["label"])
            adj = CFG["VAL_RATIO"]/(CFG["TRAIN_RATIO"]+CFG["VAL_RATIO"])
            tr,va = train_test_split(tv,test_size=adj,random_state=seed,stratify=tv["label"])
        except:
            tv,te = train_test_split(sub,test_size=CFG["TEST_RATIO"],random_state=seed)
            adj = CFG["VAL_RATIO"]/(CFG["TRAIN_RATIO"]+CFG["VAL_RATIO"])
            tr,va = train_test_split(tv,test_size=adj,random_state=seed)
        tr_l.append(tr);va_l.append(va);te_l.append(te)
    return (pd.concat(tr_l,ignore_index=True),
            pd.concat(va_l,ignore_index=True),
            pd.concat(te_l,ignore_index=True))
 
def scale_no_leakage(tr,va,te,fc):
    sc = StandardScaler()
    sc.fit(tr[fc].values.astype(np.float64))
    ts,vs,tes = tr.copy(),va.copy(),te.copy()
    for df_out,df_in in [(ts,tr),(vs,va),(tes,te)]:
        df_out[fc] = sc.transform(df_in[fc].values.astype(np.float64)).astype(np.float32)
    return ts,vs,tes,sc
 
def compute_client_class_weights(train_df):
    weights = {}
    for dev in sorted(train_df["device_id"].unique()):
        sub = train_df[train_df["device_id"]==dev]
        n_ben = int((sub["label"]==0).sum()); n_att = int((sub["label"]==1).sum())
        if n_ben==0 or n_att==0: weights[dev] = torch.tensor([1.0,1.0],dtype=torch.float32)
        else: weights[dev] = torch.tensor([min(n_att/n_ben,50.0),1.0],dtype=torch.float32)
    return weights
 
print("Splitting and scaling (seed=42) …")
TRAIN_DF,VAL_DF,TEST_DF = split_per_device(RAW_DF,seed=42)
TRAIN_DF,VAL_DF,TEST_DF,SCALER = scale_no_leakage(TRAIN_DF,VAL_DF,TEST_DF,FEAT_COLS)
del RAW_DF; gc.collect()
CLIENT_CLASS_WEIGHTS = compute_client_class_weights(TRAIN_DF)
TEST_DS = TensorDataset(
    torch.tensor(TEST_DF[FEAT_COLS].values,dtype=torch.float32),
    torch.tensor(TEST_DF["label"].values,dtype=torch.long),
)
TEST_LOADER = DataLoader(TEST_DS,batch_size=2048,shuffle=False,
                          num_workers=0,pin_memory=(DEVICE.type=="cuda"))
print(f"Test: {len(TEST_DS):,} samples")
print("✓ Cell 3 complete")

Splitting and scaling (seed=42) …
Test: 67,500 samples
✓ Cell 3 complete


In [6]:
# ===== CELL 4: Model =====
 
class IDSModel(nn.Module):
    def __init__(self,input_dim=None,h1=128,h2=64,n_classes=2):
        super().__init__()
        if input_dim is None: input_dim = INPUT_DIM
        self.net = nn.Sequential(
            nn.Linear(input_dim,h1),nn.ReLU(),
            nn.Linear(h1,h2),nn.ReLU(),
            nn.Linear(h2,n_classes),
        )
    def forward(self,x): return self.net(x)
 
_t = IDSModel()
assert sum(v.numel() for v in _t.state_dict().values()) == 23_234
assert len(list(_t.buffers())) == 0
del _t
print("IDSModel: 23,234 params ✓")
print("✓ Cell 4 complete")

IDSModel: 23,234 params ✓
✓ Cell 4 complete


In [7]:
# ===== CELL 5: FL Utilities =====
 
def sd_to_np(sd):
    return np.concatenate([v.detach().cpu().numpy().flatten() for v in sd.values()]).astype(np.float64)
 
def np_to_sd(vec,template):
    out={}; idx=0
    for k,v in template.items():
        n=v.numel()
        out[k]=torch.from_numpy(vec[idx:idx+n].reshape(v.shape).astype(np.float32))
        idx+=n
    return out
 
def create_client_loaders(train_df,feat_cols,batch_size,seed):
    loaders,sizes=[],[]
    for dev in sorted(train_df["device_id"].unique()):
        sub=train_df[train_df["device_id"]==dev]
        ds=TensorDataset(torch.tensor(sub[feat_cols].values,dtype=torch.float32),
                         torch.tensor(sub["label"].values,dtype=torch.long))
        gen=torch.Generator(); gen.manual_seed(int(seed)+int(dev))
        dl=DataLoader(ds,batch_size=batch_size,shuffle=True,generator=gen,
                      drop_last=False,num_workers=0,pin_memory=(DEVICE.type=="cuda"))
        loaders.append(dl); sizes.append(len(ds))
    return loaders,sizes
 
def train_local(global_sd,loader,epochs,lr,device,flip_labels=False,class_weights=None):
    model=IDSModel(input_dim=INPUT_DIM).to(device)
    model.load_state_dict({k:v.to(device) for k,v in global_sd.items()})
    model.train()
    opt=torch.optim.Adam(model.parameters(),lr=lr)
    crit=nn.CrossEntropyLoss(weight=class_weights.to(device) if class_weights is not None else None)
    for _ in range(epochs):
        for Xb,yb in loader:
            Xb=Xb.to(device); yb=yb.to(device)
            if flip_labels: yb=1-yb
            opt.zero_grad(); crit(model(Xb),yb).backward(); opt.step()
    return {k:v.cpu() for k,v in model.state_dict().items()}
 
def evaluate_global(global_sd,test_loader,device):
    model=IDSModel(input_dim=INPUT_DIM).to(device)
    model.load_state_dict({k:v.to(device) for k,v in global_sd.items()})
    model.eval()
    all_p,all_y=[],[]
    with torch.no_grad():
        for Xb,yb in test_loader:
            all_p.extend(model(Xb.to(device)).argmax(dim=1).cpu().numpy())
            all_y.extend(yb.numpy())
    cm=confusion_matrix(all_y,all_p,labels=[0,1])
    tn,fp,fn,tp=cm.ravel()
    asr=(fn/(fn+tp))*100 if (fn+tp)>0 else 0.0
    fpr=(fp/(fp+tn))*100 if (fp+tn)>0 else 0.0
    return {"asr":asr,"fpr":fpr,"balanced_accuracy":((100-asr)+(100-fpr))/2.0,
            "tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp)}
 
def init_global_model(seed):
    set_all_seeds(seed)
    return {k:v.cpu() for k,v in IDSModel(input_dim=INPUT_DIM).state_dict().items()}
 
print("✓ Cell 5 complete")

✓ Cell 5 complete


In [8]:
# ===== CELL 6: Aggregation Primitives =====
 
def _compute_and_clip_deltas(client_sds,global_sd):
    deltas=[{k:csd[k].float()-global_sd[k].float() for k in global_sd} for csd in client_sds]
    vecs=[sd_to_np(d) for d in deltas]
    norms=np.array([np.linalg.norm(v) for v in vecs])
    clip_val=float(np.median(norms))
    clipped=[v*(clip_val/n) if clip_val>0.0 and n>clip_val else v.copy() for v,n in zip(vecs,norms)]
    return clipped,deltas
 
def _weiszfeld(vecs,tau,max_iters,tol,nu):
    tau_sum=tau.sum()
    if tau_sum<1e-12: tau=np.ones(len(vecs),dtype=np.float64); tau_sum=float(len(vecs))
    w=np.sum([t*v for t,v in zip(tau,vecs)],axis=0)/tau_sum
    for _ in range(max_iters):
        dists=np.maximum(np.array([np.linalg.norm(w-v) for v in vecs]),nu)
        w_new=(np.sum([t*v/d for t,v,d in zip(tau,vecs,dists)],axis=0)/np.sum(tau/dists))
        if np.linalg.norm(w_new-w)<tol: break
        w=w_new
    return w
 
print("✓ Cell 6 complete")

✓ Cell 6 complete


In [9]:
# ===== CELL 7: ACCCS-GM Trust Engine =====
 
class AccumulatingCosineTrust:
    """
    Accumulating Cross-Client Cosine Similarity (ACCCS) trust scoring.
 
    [FIX-1] warmup=ACCCS_WARMUP (3) — matches the published main experiment.
    The original sensitivity notebook used warmup=1, making sensitivity results
    incomparable to the main ACCCS-GM result (which used warmup=3).
 
    Warmup rounds 0 to warmup-1: plain GM (uniform trust).
    From round warmup onward: ACCCS trust scores applied.
 
    Why warmup=3?
      Round 0: no reference (plain GM). Aggregate stored as ref.
      Round 1: first reference available but contaminated by Byzantine
               clients at full weight (4/9 = 44%). Signal weak.
      Round 2: second reference. Honest majority (5/9) has had 2 rounds
               to dominate the aggregate direction.
      Round 3+: reference is reliably honest-biased. ACCCS activates.
    """
    def __init__(self, n_clients, decay=0.95, gamma=5.0, tau_min=0.15,
                 warmup=ACCCS_WARMUP):
        assert 0 < decay <= 1.0 and gamma > 0 and 0 < tau_min < 1
        assert warmup >= 1
        self.n       = n_clients
        self.decay   = decay
        self.gamma   = gamma
        self.tau_min = tau_min
        self.warmup  = warmup
        self._accum  = np.zeros(n_clients, dtype=np.float64)
        self._prev   = None
        self._round  = 0
 
    def set_reference(self, agg_delta_vec):
        """Call AFTER aggregation each round."""
        self._prev  = agg_delta_vec.copy()
        self._round += 1
 
    def compute_trusts(self, delta_vecs):
        n = len(delta_vecs)
        assert n == self.n
        if self._round < self.warmup or self._prev is None:
            return np.ones(n, dtype=np.float64)
        ref_norm = np.linalg.norm(self._prev) + 1e-12
        if ref_norm < 1e-8:
            return np.ones(n, dtype=np.float64)
        self._accum *= self.decay
        for i, delta in enumerate(delta_vecs):
            d_norm  = np.linalg.norm(delta) + 1e-12
            cos_sim = float(np.dot(delta, self._prev) / (d_norm * ref_norm))
            self._accum[i] += cos_sim
        median_s = float(np.median(self._accum))
        tau = 1.0 / (1.0 + np.exp(-self.gamma * (self._accum - median_s)))
        return np.maximum(self.tau_min, tau).astype(np.float64)
 
 
def acccs_gm(client_sds, global_sd, acccs_engine):
    """
    ACCCS-weighted geometric median.
    Trust computed on unclipped deltas (scale-invariant).
    Weiszfeld applied to clipped deltas.
    Returns (agg_sd, agg_delta_vec) — caller must call set_reference(agg_delta_vec).
    """
    delta_vecs, delta_dicts = [], []
    for csd in client_sds:
        d = {k: csd[k].float()-global_sd[k].float() for k in global_sd}
        delta_dicts.append(d)
        delta_vecs.append(sd_to_np(d))
 
    tau = acccs_engine.compute_trusts(delta_vecs)
 
    norms    = np.array([np.linalg.norm(v) for v in delta_vecs])
    clip_val = float(np.median(norms))
    clipped  = [v*(clip_val/n) if clip_val>0.0 and n>clip_val else v.copy()
                for v,n in zip(delta_vecs,norms)]
 
    if tau.sum() < 1e-12: tau = np.ones(len(clipped), dtype=np.float64)
    w     = _weiszfeld(clipped, tau, CFG["GM_MAX_ITERS"], CFG["GM_TOL"], CFG["GM_NU"])
    agg_d = np_to_sd(w, delta_dicts[0])
    agg_sd = {k: global_sd[k].float()+agg_d[k] for k in global_sd}
    return agg_sd, w
 
 
# Verify warmup fix
_acc_test = AccumulatingCosineTrust(n_clients=9, decay=0.95, gamma=5.0,
                                     tau_min=0.15, warmup=ACCCS_WARMUP)
_tau0 = _acc_test.compute_trusts([np.random.randn(23234) for _ in range(9)])
print(f"ACCCS [FIX-1] warmup={ACCCS_WARMUP}: round 0 trust all 1.0? "
      f"{'YES ✓' if np.allclose(_tau0, 1.0) else 'NO ✗'}")
del _acc_test, _tau0
print("✓ Cell 7 complete — ACCCS-GM defined with warmup=3")

ACCCS [FIX-1] warmup=3: round 0 trust all 1.0? YES ✓
✓ Cell 7 complete — ACCCS-GM defined with warmup=3


In [10]:
# ===== CELL 8: Experiment Runner =====
 
def run_acccs_experiment(seed, gamma, decay):
    """
    Run one ACCCS-GM experiment with configurable gamma and decay.
    warmup=ACCCS_WARMUP=3 (fixed to match main experiment).
    """
    if torch.cuda.is_available(): torch.cuda.synchronize()
    set_all_seeds(seed)
 
    client_loaders, client_sizes = create_client_loaders(
        TRAIN_DF, FEAT_COLS, CFG["BATCH_SIZE"], seed)
    n_clients = len(client_loaders)
 
    n_byz   = round(n_clients * CFG["BYZ_RATIO"])
    byz_ids = set(np.random.RandomState(seed).choice(
        n_clients, size=n_byz, replace=False).tolist())
 
    global_sd = init_global_model(seed)
    acccs     = AccumulatingCosineTrust(
        n_clients=n_clients, decay=decay, gamma=gamma,
        tau_min=0.15, warmup=ACCCS_WARMUP,  # [FIX-1]
    )
 
    for rnd in range(CFG["FL_ROUNDS"]):
        client_updates = []
        for cid, loader in enumerate(client_loaders):
            cw = CLIENT_CLASS_WEIGHTS.get(cid, torch.tensor([1.0, 1.0]))
            sd = train_local(global_sd, loader,
                             epochs=CFG["LOCAL_EPOCHS"], lr=CFG["LR"],
                             device=DEVICE, flip_labels=(cid in byz_ids),
                             class_weights=cw)
            client_updates.append(sd)
 
        global_sd, agg_vec = acccs_gm(client_updates, global_sd, acccs)
        acccs.set_reference(agg_vec)  # MUST be called AFTER aggregation
        del client_updates
 
    return evaluate_global(global_sd, TEST_LOADER, DEVICE), sorted(byz_ids), n_byz
 
 
print("✓ Cell 8 complete — experiment runner defined")

✓ Cell 8 complete — experiment runner defined


In [ ]:
# ===== CELL 9: Run ACCCS Experiments =====
 
print("=" * 65)
print("SESSION 2: ACCCS-GM SENSITIVITY EXPERIMENTS")
print(f"  warmup_rounds = {ACCCS_WARMUP}  [FIX-1: matches main ACCCS-GM experiment]")
print("=" * 65)
 
all_results = []
t_start = time.time()
run_idx = 0
_chk_path = f"{CFG['OUT_DIR']}/sensitivity_session2_checkpoint.csv"
 
def _save_chk(label=""):
    if all_results:
        pd.DataFrame(all_results).to_csv(_chk_path, index=False)
        if label:
            print(f"  ✓ Checkpoint [{label}]: {len(all_results)} rows")
 
def _log(run_idx, n_total, p, v, seed, m):
    elapsed = time.time() - t_start
    eta_h = (elapsed / run_idx) * (n_total - run_idx) / 3600
    return (f"  [{run_idx:3d}/{n_total}] {p}={v}  seed={seed:4d}  "
            f"ASR={m['asr']:.4f}%  FPR={m['fpr']:.4f}%  ETA={eta_h:.1f}h")
 
 
# ── Experiment 5: gamma sensitivity ───────────────────────────────────────────
print(f"\n{'─'*50}")
print("EXPERIMENT 5: ACCCS γ sensitivity")
print("  Fixed: decay=0.95, byz=40%, warmup=3")
print("  Question: Is γ=5.0 robust? At what γ does ASR/FPR degrade?")
print(f"{'─'*50}")
 
for gamma_val in EXP5_GAMMA_VALS:
    for seed in CFG["SEEDS"]:
        run_idx += 1
        t0 = time.time()
        metrics, byz_ids, n_byz = run_acccs_experiment(seed, gamma=gamma_val, decay=0.95)
        elapsed = time.time() - t0
        all_results.append({
            "experiment": "exp5_gamma", "method": "acccs_gm",
            "byz_ratio": CFG["BYZ_RATIO"], "seed": seed,
            "param_name": "gamma", "param_value": gamma_val,
            "asr": metrics["asr"], "fpr": metrics["fpr"],
            "balanced_accuracy": metrics["balanced_accuracy"],
            "tn": metrics["tn"], "fp": metrics["fp"],
            "fn": metrics["fn"], "tp": metrics["tp"],
            "elapsed_s": round(elapsed, 1),
        })
        gc.collect()
        print(_log(run_idx, n_runs_session2, "gamma", f"{gamma_val:5.1f}", seed, metrics))
    _save_chk(f"Exp5 gamma={gamma_val}")
 
print("\n  Exp5 complete. Per-gamma summary:")
_exp5 = pd.DataFrame([r for r in all_results if r["experiment"]=="exp5_gamma"])
for v in sorted(_exp5["param_value"].unique()):
    s = _exp5[_exp5["param_value"]==v]
    print(f"    γ={v:5.1f}: ASR={s['asr'].mean():.4f}%±{s['asr'].std():.4f}%  "
          f"FPR={s['fpr'].mean():.4f}%±{s['fpr'].std():.4f}%")
 
 
# ── Experiment 6: decay sensitivity ───────────────────────────────────────────
print(f"\n{'─'*50}")
print("EXPERIMENT 6: ACCCS decay sensitivity")
print("  Fixed: gamma=5.0, byz=40%, warmup=3")
print("  [FIX-2] BST prior-decay bug corrected (irrelevant here — ACCCS only)")
print("  Question: Does exponential decay (0.95) outperform full accumulation (1.0)?")
print(f"{'─'*50}")
 
for decay_val in EXP6_DECAY_VALS:
    for seed in CFG["SEEDS"]:
        run_idx += 1
        t0 = time.time()
        metrics, byz_ids, n_byz = run_acccs_experiment(seed, gamma=5.0, decay=decay_val)
        elapsed = time.time() - t0
        hl_str = f"{math.log(0.5)/math.log(decay_val):.1f}" if decay_val < 1.0 else "∞"
        all_results.append({
            "experiment": "exp6_decay", "method": "acccs_gm",
            "byz_ratio": CFG["BYZ_RATIO"], "seed": seed,
            "param_name": "decay", "param_value": decay_val,
            "asr": metrics["asr"], "fpr": metrics["fpr"],
            "balanced_accuracy": metrics["balanced_accuracy"],
            "tn": metrics["tn"], "fp": metrics["fp"],
            "fn": metrics["fn"], "tp": metrics["tp"],
            "elapsed_s": round(elapsed, 1),
        })
        gc.collect()
        print(_log(run_idx, n_runs_session2, "decay", f"{decay_val:.2f}", seed, metrics))
    _save_chk(f"Exp6 decay={decay_val}")
 
print("\n  Exp6 complete. Per-decay summary:")
_exp6 = pd.DataFrame([r for r in all_results if r["experiment"]=="exp6_decay"])
for v in sorted(_exp6["param_value"].unique()):
    s = _exp6[_exp6["param_value"]==v]
    hl = f"{math.log(0.5)/math.log(v):.1f}" if v < 1.0 else "∞"
    print(f"    decay={v:.2f} (half-life≈{hl} rds): ASR={s['asr'].mean():.4f}%  "
          f"FPR={s['fpr'].mean():.4f}%")
 
 
# ── Save Session 2 CSV ────────────────────────────────────────────────────────
df2 = pd.DataFrame(all_results)
csv2_path = f"{CFG['OUT_DIR']}/sensitivity_session2.csv"
df2.to_csv(csv2_path, index=False)
if Path(_chk_path).exists(): Path(_chk_path).unlink()
total_h = (time.time() - t_start) / 3600
print(f"\n{'='*65}")
print(f"SESSION 2 COMPLETE in {total_h:.2f} hours")
print(f"Total runs: {run_idx}")
print(f"Saved: {csv2_path}  ({len(df2)} rows)")
print("✓ Cell 9 complete")

SESSION 2: ACCCS-GM SENSITIVITY EXPERIMENTS
  warmup_rounds = 3  [FIX-1: matches main ACCCS-GM experiment]

──────────────────────────────────────────────────
EXPERIMENT 5: ACCCS γ sensitivity
  Fixed: decay=0.95, byz=40%, warmup=3
  Question: Is γ=5.0 robust? At what γ does ASR/FPR degrade?
──────────────────────────────────────────────────
  [  1/81] gamma=  1.0  seed=  42  ASR=0.0274%  FPR=0.6823%  ETA=4.3h
  [  2/81] gamma=  1.0  seed= 123  ASR=0.0194%  FPR=2.0111%  ETA=4.2h
  [  3/81] gamma=  1.0  seed= 456  ASR=0.0194%  FPR=1.9932%  ETA=4.1h
  [  4/81] gamma=  1.0  seed= 789  ASR=0.2131%  FPR=1.2929%  ETA=4.0h
  [  5/81] gamma=  1.0  seed=1011  ASR=0.0355%  FPR=0.3950%  ETA=3.9h
  [  6/81] gamma=  1.0  seed=1415  ASR=0.0258%  FPR=1.1133%  ETA=3.9h
  [  7/81] gamma=  1.0  seed=1617  ASR=0.0258%  FPR=0.8619%  ETA=3.8h
  [  8/81] gamma=  1.0  seed=1819  ASR=0.0194%  FPR=2.5857%  ETA=3.7h
  [  9/81] gamma=  1.0  seed=2021  ASR=0.0194%  FPR=2.3164%  ETA=3.7h
  ✓ Checkpoint [Exp5 gamma

In [ ]:
# ===== CELL 10: Combined Analysis (Session 1 + Session 2) =====
 
print("\n" + "="*65)
print("COMBINED ANALYSIS (SESSIONS 1 + 2)")
print("="*65)
 
# Exp 5: gamma analysis
print("\nExperiment 5: γ Sensitivity")
print(f"  Main experiment (γ=5.0, warmup=3): ASR=0.0249%  FPR=1.1612%")
print(f"  Reference: Trust-GM: 0.0728%  |  Plain GM: 1.9658%")
print(f"\n  {'γ':>6}  {'Mean ASR':>10}  {'Std ASR':>10}  {'Max ASR':>10}  {'Mean FPR':>10}")
print("  " + "-" * 50)
for v in sorted(_exp5["param_value"].unique()):
    s = _exp5[_exp5["param_value"]==v]
    mark = " ← default" if abs(v-5.0) < 0.01 else ""
    print(f"  {v:>6.1f}  {s['asr'].mean():>10.4f}%  {s['asr'].std():>10.4f}%  "
          f"{s['asr'].max():>10.4f}%  {s['fpr'].mean():>10.4f}%{mark}")
 
# Exp 6: decay analysis
print("\nExperiment 6: Decay Sensitivity")
print(f"  Main experiment (decay=0.95, warmup=3): ASR=0.0249%  FPR=1.1612%")
print(f"\n  {'decay':>7}  {'half-life':>10}  {'Mean ASR':>10}  {'Std ASR':>10}  {'Mean FPR':>10}")
print("  " + "-" * 55)
for v in sorted(_exp6["param_value"].unique()):
    s  = _exp6[_exp6["param_value"]==v]
    hl = f"{math.log(0.5)/math.log(v):.1f}" if v < 1.0 else "  ∞"
    mark = " ← default" if abs(v-0.95) < 0.01 else ""
    print(f"  {v:>7.2f}  {hl:>10}  {s['asr'].mean():>10.4f}%  {s['asr'].std():>10.4f}%  "
          f"{s['fpr'].mean():>10.4f}%{mark}")
 
# Combined Session 1 + Session 2 summary
if len(SESSION1_DF) > 0:
    all_df = pd.concat([SESSION1_DF, df2], ignore_index=True)
    all_df.to_csv(f"{CFG['OUT_DIR']}/sensitivity_combined.csv", index=False)
    print(f"\nCombined CSV: {CFG['OUT_DIR']}/sensitivity_combined.csv  ({len(all_df)} rows)")
    print("\nExperiment counts:")
    print(all_df["experiment"].value_counts().to_string())
 
print("✓ Cell 10 complete")

In [ ]:
# ===== CELL 11: Statistical Tests (Session 2) =====
 
if HAVE_SCIPY:
    seeds     = CFG["SEEDS"]
    tgm_vals  = [TRUST_GM_40PCT_PER_SEED[s] for s in seeds]
    gm_vals   = [GM_40PCT_PER_SEED[s] for s in seeds]
    acccs_ref = [ACCCS_GM_40PCT_PER_SEED[s] for s in seeds]
 
    print("\n" + "="*65)
    print("STATISTICAL TESTS: ACCCS-GM sensitivity (Holm-Bonferroni)")
    print("="*65)
 
    all_pvals = []
 
    # ACCCS reference (main experiment, gamma=5, decay=0.95, warmup=3) vs Trust-GM
    try:
        stat, p = sp_stats.wilcoxon(acccs_ref, tgm_vals, alternative="less")
        p = p if np.isfinite(p) else 1.0
    except: p = 1.0
    all_pvals.append(("ACCCS-GM(main) vs Trust-GM", p))
 
    # gamma sweep vs Trust-GM
    for g_val in EXP5_GAMMA_VALS:
        sub = _exp5[_exp5["param_value"]==g_val]
        if len(sub) == 9:
            v = [sub[sub["seed"]==s]["asr"].values[0] for s in seeds]
            try:
                stat, p = sp_stats.wilcoxon(v, tgm_vals, alternative="less")
                p = p if np.isfinite(p) else 1.0
            except: p = 1.0
            all_pvals.append((f"ACCCS-GM(γ={g_val:.1f}) vs Trust-GM", p))
 
    # decay sweep vs Trust-GM
    for d_val in EXP6_DECAY_VALS:
        sub = _exp6[_exp6["param_value"]==d_val]
        if len(sub) == 9:
            v = [sub[sub["seed"]==s]["asr"].values[0] for s in seeds]
            try:
                stat, p = sp_stats.wilcoxon(v, tgm_vals, alternative="less")
                p = p if np.isfinite(p) else 1.0
            except: p = 1.0
            all_pvals.append((f"ACCCS-GM(decay={d_val:.2f}) vs Trust-GM", p))
 
    # Holm-Bonferroni
    all_pvals_sorted = sorted(all_pvals, key=lambda x: x[1])
    m = len(all_pvals_sorted); alpha = 0.05
    print(f"\n  m = {m} simultaneous tests, α = {alpha}")
    print(f"  {'Rank':<4} {'Comparison':<42} {'p':>9}  {'Threshold':>10}  {'Result'}")
    print("  " + "-" * 73)
    all_pass = True
    for i, (name, p) in enumerate(all_pvals_sorted, 1):
        threshold = alpha / (m - i + 1)
        result = "PASS ✓" if p < threshold else "FAIL ✗"
        if p >= threshold: all_pass = False
        print(f"  {i:<4} {name:<42} {p:>9.5f}  {threshold:>10.5f}  {result}")
    print(f"\n  All tests survive Holm-Bonferroni: {'YES ✓' if all_pass else 'NO ✗'}")
 
print("✓ Cell 11 complete")
 

In [ ]:
# ===== CELL 12: Visualization =====
 
if HAVE_PLOT:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle("Session 2: ACCCS-GM Sensitivity Analysis (40% Byzantine)",
                 fontsize=13, fontweight="bold")
 
    # Plot 1: gamma sensitivity
    ax = axes[0]
    g_vals  = sorted(_exp5["param_value"].unique())
    m_asr   = [_exp5[_exp5["param_value"]==v]["asr"].mean() for v in g_vals]
    s_asr   = [_exp5[_exp5["param_value"]==v]["asr"].std()  for v in g_vals]
    m_fpr   = [_exp5[_exp5["param_value"]==v]["fpr"].mean() for v in g_vals]
    ax2     = ax.twinx()
    ax.errorbar(g_vals, m_asr, yerr=s_asr, marker="o", color="#8E24AA",
                lw=2, capsize=5, label="ASR ± std (left)")
    ax2.plot(g_vals, m_fpr, marker="s", color="#FB8C00", ls="--", lw=2, label="FPR (right)")
    ax.axhline(y=0.0728, color="#43A047", ls="--", lw=1.2, alpha=0.7, label="Trust-GM ASR")
    ax.set_xlabel("Sigmoid sharpness γ")
    ax.set_ylabel("Mean ASR (%)", color="#8E24AA")
    ax2.set_ylabel("Mean FPR (%)", color="#FB8C00")
    ax.set_title(f"Exp 5: γ Sensitivity (ACCCS-GM)\nwarmup={ACCCS_WARMUP}, decay=0.95")
    ax.grid(True, alpha=0.3)
    lines1, _ = ax.get_legend_handles_labels()
    lines2, _ = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, [l.get_label() for l in lines1 + lines2], fontsize=8)
 
    # Plot 2: decay sensitivity
    ax = axes[1]
    d_vals  = sorted(_exp6["param_value"].unique())
    m_asr_d = [_exp6[_exp6["param_value"]==v]["asr"].mean() for v in d_vals]
    s_asr_d = [_exp6[_exp6["param_value"]==v]["asr"].std()  for v in d_vals]
    m_fpr_d = [_exp6[_exp6["param_value"]==v]["fpr"].mean() for v in d_vals]
    ax3     = ax.twinx()
    ax.errorbar([str(v) for v in d_vals], m_asr_d, yerr=s_asr_d,
                marker="o", color="#8E24AA", lw=2, capsize=5, label="ASR ± std (left)")
    ax3.plot([str(v) for v in d_vals], m_fpr_d, marker="s", color="#FB8C00",
             ls="--", lw=2, label="FPR (right)")
    ax.axhline(y=0.0728, color="#43A047", ls="--", lw=1.2, alpha=0.7, label="Trust-GM ASR")
    ax.set_xlabel("Decay factor")
    ax.set_ylabel("Mean ASR (%)", color="#8E24AA")
    ax3.set_ylabel("Mean FPR (%)", color="#FB8C00")
    ax.set_title(f"Exp 6: Decay Sensitivity (ACCCS-GM)\nwarmup={ACCCS_WARMUP}, γ=5.0")
    ax.grid(True, alpha=0.3)
 
    plt.tight_layout()
    fig_path = f"{CFG['OUT_DIR']}/sensitivity_session2.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Figure: {fig_path}")
 
print("=" * 65)
print("SESSION 2 COMPLETE")
print(f"CSV:    {csv2_path}")
print(f"Runtime: {(time.time()-t_start)/3600:.2f} hours")
print()
print("Download from /kaggle/working/:")
print("  sensitivity_session2.csv")
print("  sensitivity_combined.csv   (if Session 1 CSV was available)")
print("  sensitivity_session2.png")
print("=" * 65)
print("✓ Cell 12 complete — all done")
# This notebook is fully runnable on Kaggle without modification.
 